In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from dotenv import load_dotenv
import os 
load_dotenv()

openai_client = OpenAI()

groq_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [ ]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant_openai = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
    model='gpt-5.4-mini',
    api_type='openai'
)

assistant_groq = RAGBase(
    index=index,
    llm_client=groq_client,
    instructions=instructions,
    model='qwen/qwen3.6-27b',
    api_type='groq'
)

In [ ]:
answer = assistant_openai.rag('How do I run Ollama locally?')
print("OpenAI Answer:")
print(answer)

answer = assistant_groq.rag('How do I run Ollama locally?')
print("Groq Answer:")
print(answer)



In [38]:
messages = [
    {"role": "user", "content": "How do I run Olama locally?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

print(response.output_text)


I’m assuming you mean **Ollama** (often misspelled “Olama”). Here’s the quickest way to run it locally.

## 1) Install Ollama
### macOS
```bash
brew install ollama
```
Or download it from:
https://ollama.com/download

### Linux
```bash
curl -fsSL https://ollama.com/install.sh | sh
```

### Windows
Download and install from:
https://ollama.com/download

## 2) Start the Ollama service
Usually it starts automatically, but if needed:

```bash
ollama serve
```

## 3) Download and run a model
For example, run Llama 3:

```bash
ollama run llama3
```

This will download the model the first time, then open an interactive chat.

Other examples:
```bash
ollama run mistral
ollama run phi3
ollama run gemma
```

## 4) Use it from the API
Ollama runs a local API at:

```bash
http://localhost:11434
```

Example request:
```bash
curl http://localhost:11434/api/generate -d '{
  "model": "llama3",
  "prompt": "Write a haiku about local AI."
}'
```

## 5) Check installed models
```bash
ollama list
```

##

In [39]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [40]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for answers to user questions",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [41]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

print(response.output)

[ResponseFunctionToolCall(arguments='{"query":"How do I run Olama locally? Ollama local install run model start service FAQ"}', call_id='call_nZcAPnTvT0vMZ0gH50qYS0IF', name='search', type='function_call', id='fc_05007a37030f690b006a92c9000d9087d28480e33294b5ca5f', caller=None, namespace=None, status='completed')]


In [44]:
import json
call = response.output[0]
args = json.loads(call.arguments)
print("Arguments:", args)

results = search(**args)
result_json = json.dumps(results, indent=2)
print("Search Results:", result_json)

Arguments: {'query': 'How do I run Olama locally? Ollama local install run model start service FAQ'}
Search Results: [
  {
    "id": "1d0b969028",
    "course": "llm-zoomcamp",
    "section": "Module 1: RAG",
    "question": "Ollama: How to install Ollama?",
    "answer": "First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a respo

In [ ]:
messages.extend(response.output)

messages.append({
    "type" : "function_call_output",
    "call_id" : call.call_id,
    "output" : result_json
})


In [ ]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

print(response.output_text)

In [ ]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "HOw to make coffee ?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)



In [ ]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [ ]:
agent_loop(instructions, question)